# Foundation-model evaluation

Onceover's detectors are a first pass over an unfamiliar table. A second
pass could go further for a caller who names a target: predict the target
from several related tables, or forecast a measure along the frame's time
axis. Pretrained foundation models promise both without task-specific
training. This appendix records the evaluation that tested that promise
before any screener was designed on top of it.

The evaluation ran five experiments over two rounds on 2026-11-03, plus a
completion probe on 2026-11-05. Round 1
ran NVIDIA's Kumo Relational, a pretrained model that predicts a label
from linked tables, and Mistral's Timestral, a family of pretrained
zero-shot forecasting models. Round 2 ran the remaining models in
NVIDIA's Kumo family under the same protocols: Kumo Forecast, Kumo
Anomaly, and Kumo Tabular. Every experiment ran on CPU on one machine,
against baselines computed on the same data, and this appendix shows the
protocols, the deviations from them, and the measured results — including
one experiment that produced no result at its shipped settings and was
completed later at reduced sampling.

The design that consumes this evidence treats both model families as
optional second-pass screeners: opt-in, gated on a caller-named target,
and required to report every statement with its baseline attached. The
closing sections state what the measurements establish, what they leave
open, and where the scripts, logs, and raw outputs live.

## Environment

Every run in both rounds executed on one virtual machine: Ubuntu
24.04.5 LTS on x86_64, with 4 cores, 7,934 MiB of RAM, and no swap. The
machine has no GPU. That was verified directly rather than inferred from
a specification sheet: no `/dev/nvidia*` device exists, `nvidia-smi` is
not installed, and `torch.cuda.is_available()` returned False in every
environment the probes built. Whether a GPU was required at all is part
of the Experiment 1 record below.

Each experiment ran in its own Python 3.12.3 virtual environment, and
the stacks differ because the models pin different dependencies:

| Experiment | Stack | Checkpoint |
| --- | --- | --- |
| 1 — Kumo Relational | torch 2.14.0+cpu; pyg-lib 0.9.0+pt214cpu; structured-data-models 0.2.0rc2.dev214+gc51bc3752, installed from NVIDIA's GitHub repository at commit c51bc3752; numpy 2.5.4; pandas 3.1.0; pyarrow 26.0.1; huggingface_hub 2.4.0; safetensors 0.8.0; scikit-learn 1.9.1 | `nvidia/Kumo-Relational`, revision v1.0.1: `classifier.pt` 119,862,733 bytes; `regressor.pt` 123,786,699 bytes |
| 2 — Timestral | torch 2.15.0+cpu; timestral 1.0.0, installed from Mistral's GitHub repository; numpy 2.5.4; polars 2.1.0; safetensors 0.8.0; huggingface_hub 2.4.0 | `mistralai/Timestral-28M`, 111,296,464 bytes; `mistralai/Timestral-5M`, 20,843,960 bytes |
| 3 — Kumo Forecast | torch 2.14.0+cpu; transformers 5.22.1; pandas 3.1.0; numpy 2.5.4; Kumo-Forecast installed editable from NVIDIA's Kumo-TS repository, v1.0.0 | `run8_best_model_cr.pt`, 1,421,225,659 bytes, plus `standardizer.pkl`; MOMENT-1-large backbone, 1,385,501,256 bytes |
| 4 — Kumo Anomaly | torch 2.14.0+cpu; pandas 2.3.3; numpy 1.26.4 (the package pins numpy below 2); scikit-learn 1.9.1; scipy 1.17.1; the same Kumo-TS repository | `final_model.pth`, 29,600,287 bytes, plus `curriculum_medium.yaml` |
| 5 — Kumo Tabular | the Experiment 1 environment (structured-data-models 0.2.0rc2 at commit c51bc3752) | `nvidia/Kumo-Tabular`, revision v1.0.1, downloaded by the library |

Peak memory was measured with `resource.getrusage` (the `ru_maxrss`
field) because `/usr/bin/time` does not exist on this machine. The
machine's `/tmp` is a 512 MB tmpfs, which shaped the install workarounds
in the next section.

### Downloads and installs

Three download paths on this machine stall in ways that shaped every
setup, and a reader reproducing these runs will meet all three.

- pip could not fetch the 196 MB torch CPU wheel. One attempt failed
  after 156.8 MB of 196.3 MB with "not enough bytes received", and a
  resume-retry attempt stalled at 145.8 MB. The working route was a
  resumable `curl -L -C -` download, a sha256 verification, and a local
  install of the fetched file.
- Hugging Face's default xet transfer path stalled repeatedly: a
  139-byte config file would arrive while the weights sat at zero
  bytes. Setting `HF_HUB_DISABLE_XET=1` made the small checkpoints
  download in seconds (23.8 s for the Timestral-28M weights, 8.3 s for
  the 5M). The two large checkpoints still stalled in-process — the
  MOMENT-1-large backbone died at 84% and at 70% on two attempts — and
  were fetched with the same curl resume loop, then seeded into the
  local cache or passed to the model as a local directory.
- pip's unpack step needs more room than the 512 MB `/tmp` provides for
  the 539 MB CUDA torch wheel that one resolver run selected by
  mistake, so installs ran with `TMPDIR` pointed at the workspace disk
  and at the CPU wheel index.

No library code was patched in any setup. Where a checkpoint was seeded
by hand, the bytes are the published ones; each experiment's section
records its substitutions.

## Experiment 1 — Kumo Relational: prediction from linked tables

Kumo Relational is a pretrained model that predicts a target for rows of
one table using labeled example rows — the context — together with the
tables linked to them by foreign keys. The model is not trained on the
task; the context rows are its only task-specific input. The first
question for this experiment was narrower than accuracy: whether the
model runs at all on a CPU-only machine through the open
`structured-data-models` library, and whether its predictions on a task
with known signal are sane. Three probes answered it, all with
`device="cpu"` and 4 torch threads.

### Probe 1: a direct forward pass on a planted signal

The probe built two tables: `users` (600 rows; `user_id`, `age`) and
`orders` (2,493 rows; `user_id` as a foreign key, `amount`, `days_ago`),
plus a task table (`user_id`, timestamp, `churn`). The signal was
planted: churn probability rises as a user's latent engagement — and
therefore order volume — falls, so the signal exists only across the
join. The task gave 400 users as labeled context and 200 as queries,
with `num_hops=2` of relational context.

| Measurement | Value |
| --- | --- |
| Library import | 3.61 s |
| Model construction and checkpoint load | 2.87 s |
| Forward pass (400 context, 200 query) | 4.43 s |
| Output | one probability pair per query row; every row sums to 1.0000; values in [0.186, 0.814] |
| Holdout AUROC | 0.628 |
| Base-rate-only AUROC | 0.500 |
| Holdout base rate | 0.370 |
| Accuracy at a 0.5 threshold | 0.630, exactly the always-negative rate |
| Peak memory | 728 MiB |

AUROC is the probability that a randomly chosen positive row scores
above a randomly chosen negative one; 0.5 is chance. The 0.628 against
the base rate's 0.500 says the model recovered part of a signal that no
single table contains. Accuracy at a fixed threshold is not the
informative figure here — it equals the rate from predicting the
majority class for every row — so AUROC is what the probes report.

### Probe 2: the neighbor-sampling path

The same data was rebuilt as a `RelationalData` object and run through
the library's pyg-lib sampler path with neighbor widths [4, 4], then fit
and predict.

| Measurement | Value |
| --- | --- |
| Sampler over the context | 0.03 s |
| Sampled subgraphs | 400 users; orders pruned from 2,493 rows to 1,260 |
| Fit | 2.26 s |
| Query sampling and predict | 0.99 s |
| Holdout AUROC | 0.572 |
| Peak memory | 739 MiB |

The score fell from 0.628 to 0.572 because the [4, 4] sampling truncated
each user's order history: the evidence the model could see was smaller.
Sampling width is therefore part of a result's protocol, not an
implementation detail.

### Probe 3: AdventureWorks orders and customers

The corpus holds AdventureWorks as a single table (SalesOrderHeader,
31,465 rows), so the probe derived two tables using the source schema's
real foreign key: `orders` (primary key `sales_order_id`, foreign key
`customer_id`, with subtotal, tax amount, freight, and total due) and
`customers` (primary key `customer_id`, territory taken from first
sighting). The task was predicting each order's `online_order_flag`. The
first 1,800 orders by date supplied 1,200 context rows and 600 query
rows, with `num_hops=2`.

| Measurement | Value |
| --- | --- |
| Forward pass (1,200 context, 600 query) | 8.26 s |
| Holdout AUROC | 1.000 |
| Holdout base rate | 0.802 |
| Peak memory | 745 MiB |

The perfect score is real separability in this slice, not a target leak:
`online_order_flag` was not an input column. In the query split, freight
on online orders maxes at 89.46 while freight on reseller orders reaches
2,694.26, and the single column `total_due` alone reaches AUROC 0.84 in
the inverted direction; 117 customers appear in both splits with
consistent channel behavior. This probe is a real-data smoke test — it
shows the pipeline runs end to end on corpus data — and it is not a
benchmark of relational accuracy, because the target is nearly
determined by order size.

### Evidence note: the GPU requirement

This experiment began from a claim that running the model locally
required a CUDA GPU. The claim rested on the supported-hardware list
published for NVIDIA's containerized Kumo Relational deployment and on
the CUDA marks in the library's own test suite. It did not survive the
run, and the deciding evidence is in the library itself. Its relational
sampler is CPU-only by design: `PyGLibRelationalSampler` raises an error
unless its input data is on CPU. The repository's RelBench example
selects CUDA when it is available and CPU otherwise. The test marks are
test policy, and nothing in the model path asserts CUDA. The three
probes above then ran on CPU and produced the recorded outputs.

## Experiment 2 — Timestral: zero-shot forecasting

Timestral is Mistral's family of pretrained time-series models. The
caller hands the forecaster a context series and a horizon, and it
returns a predictive distribution for each future step — 21 quantile
levels, including 0.1 and 0.9 — with no training on the series being
forecast. The probe ran the 28M and 5M checkpoints through the documented
API (`TimestralForecaster.from_pretrained`, then `forecast(context,
prediction_length=H)`), passing context as NumPy arrays: shape `(T,)`
for one series and `(1, 4, T)` for the four-state panel. The 123M and
209M checkpoints were not run.

### Scoring

Accuracy is MASE: the mean absolute error of the model's median forecast
divided by the in-sample mean absolute error of a seasonal-naive
forecast on the context. A MASE of 1.0 ties the naive; below 1.0 beats
it. The seasonal period is 12 for the monthly panel, and the hourly
experiment's MASE denominator uses season 24. The baselines forecast by
tiling the last observed seasonal cycle: lag 12 for the panel; lag 24
and lag 168 (one week) for the hourly series. Calibration is the
empirical coverage of the central 80% interval, the share of realized
values inside [q10, q90] against a nominal level of 0.80, reported with
the mean interval width.

### The state-unemployment panel

The data is the corpus state-unemployment panel in long form (one row
per state and month). The probe forecast four states — Nevada, Michigan,
California, South Dakota — from a context of January 1976 to September
2023 (573 months), at a horizon of 24 months, scored over October 2023
to September 2025. The panel's only null is October 2025, for all 51
states; the scored window ends before it, so neither the context nor the
scored window contains a null, and nothing was imputed anywhere.

Timestral-28M (load 1.68 s; the four multivariate forecasts came from a
single panel call of 0.38 s):

| Mode | State | MASE | Seasonal-naive-12 | Coverage of 80% interval | Mean interval width | Call |
| --- | --- | --- | --- | --- | --- | --- |
| univariate | NV | 0.151 | 0.150 | 1.000 | 1.65 | 0.30 s |
| univariate | MI | 0.563 | 0.622 | 0.917 | 1.77 | 0.52 s |
| univariate | CA | 0.466 | 0.656 | 1.000 | 3.08 | 0.11 s |
| univariate | SD | 0.229 | 0.292 | 1.000 | 0.73 | 0.19 s |
| multivariate | NV | 0.156 | 0.150 | 1.000 | 1.48 | (panel) |
| multivariate | MI | 0.470 | 0.622 | 1.000 | 1.96 | (panel) |
| multivariate | CA | 0.071 | 0.656 | 1.000 | 2.36 | (panel) |
| multivariate | SD | 0.184 | 0.292 | 1.000 | 0.75 | (panel) |

Timestral-5M (load 0.33 s; panel call 0.11 s):

| Mode | State | MASE | Seasonal-naive-12 | Coverage of 80% interval | Mean interval width | Call |
| --- | --- | --- | --- | --- | --- | --- |
| univariate | NV | 0.135 | 0.150 | 0.917 | 1.10 | 0.02 s |
| univariate | MI | 0.536 | 0.622 | 1.000 | 1.92 | 0.05 s |
| univariate | CA | 0.325 | 0.656 | 1.000 | 2.43 | 0.02 s |
| univariate | SD | 0.128 | 0.292 | 1.000 | 0.89 | 0.08 s |
| multivariate | NV | 0.109 | 0.150 | 0.958 | 1.47 | (panel) |
| multivariate | MI | 0.395 | 0.622 | 1.000 | 2.20 | (panel) |
| multivariate | CA | 0.124 | 0.656 | 1.000 | 2.16 | (panel) |
| multivariate | SD | 0.120 | 0.292 | 1.000 | 0.85 | (panel) |

Three patterns, as measured:

- Both checkpoints beat the lag-12 naive on Michigan, California, and
  South Dakota in both modes. Nevada is the exception at 28M: 0.151
  univariate and 0.156 multivariate against the baseline's 0.150. The
  5M beats the baseline on Nevada in both modes.
- The 5M posts the lower MASE in 7 of the 8 state-and-mode cells. The
  exception is multivariate California, where the 28M scores 0.071 —
  far better than its own univariate 0.466 on the same series and the
  5M's panel score of 0.124. Reading the four states jointly is where
  the larger checkpoint earned its largest margin.
- Coverage is far above nominal: 1.000 in 13 of the 16 cells, and
  between 0.917 and 0.958 in the remaining three, against a nominal
  0.80. On these smooth monthly series the intervals are wider than
  the realized errors require.

### Beijing PM2.5 at the Dongsi station

The data is the Beijing air-quality extract (`beijing_all.csv`, read
with `null_values=['NA']`), one station (Dongsi) only. The window runs
from 2013-08-13 00:00 to 2013-10-11 23:00 — 60 days on a fully
contiguous hourly grid. The context is the first 1,392 hours; the
forecast covers the final 48 hours, October 10–11. The window contains
exactly one PM2.5 null, at 2013-10-08 12:00, inside the context. It was
passed to the model as NaN — the library's native missing encoding —
with no imputation. The scored 48 hours are fully observed, and the
baselines' source zones are null-free.

| Checkpoint | MASE | vs naive-24 (2.069) | vs naive-168 (1.791) | Coverage of 80% interval | Mean interval width | Load | Forecast |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Timestral-28M | 1.850 | beats | loses | 0.521 | 231.3 µg/m³ | 0.96 s | 0.39 s |
| Timestral-5M | 2.198 | loses | loses | 0.438 | 233.8 µg/m³ | 0.22 s | 0.09 s |

In absolute terms the 28M's mean error was 97.5 µg/m³, against 109.1
for the daily naive and 94.5 for the weekly naive; the 5M's mean error
was 115.9 µg/m³. Both checkpoints lose to the weekly naive on this
window. Calibration fails in the opposite direction from the panel: the
intervals average about 231 µg/m³ wide and still cover only about half
the realized values.

Two earlier attempts at this experiment were aborted by bugs in the
probe script's own assertions about the baselines' source windows, not
by library failures. The assertions were corrected, and the final run
completed end to end.

## Experiment 3 — Kumo Forecast on the Experiment 2 protocol

Kumo Forecast is NVIDIA's multivariate forecasting model in the Kumo-TS
repository (v1.0.0). It pairs a MOMENT-1-large backbone — about 340
million parameters — with a forecasting checkpoint; an optional mode
retrieves historical patterns and blends them into the forecast. Its
standardizer is a single global scalar pair (mean −5.6712, standard
deviation 8.6933), so it broadcasts across any channel count. This
experiment replicated the Experiment 2 protocol exactly — same panel,
same context and scoring windows, same Beijing window, same baselines,
same MASE definition — so the two forecasters meet on identical ground.
The replication check passed: the recomputed seasonal-naive MASEs match
Experiment 2 exactly (0.150, 0.622, 0.656, 0.292 monthly; 2.069 and
1.791 hourly).

Reading the repository's README and SDK source settled the operating
mode first: `perform_forecasting()` is the zero-shot path. It downloads
its weights and forecasts without fine-tuning. A fine-tuning example
exists separately, freezing the encoder and training the head; it was
not used.

### Deviations the SDK forced

- **Context length.** The SDK consumes only the trailing 512 rows of
  the context (its `seq_len`). Timestral consumed all 573 months and
  all 1,392 hours.
- **Missing values.** The SDK zero-fills nulls. The single Beijing
  context null falls inside the trailing-512 window and entered as
  0.0; Timestral received the same value as NaN.
- **Output.** Kumo Forecast emits point forecasts only — no quantiles —
  so interval coverage is not computable for it. Timestral emits 21
  quantiles per step.
- **Horizon.** The default configuration's `model_horizon` is 72, so
  the 24- and 48-step horizons needed no autoregressive rollout.

### Kumo Forecast results

State unemployment, univariate:

| State | Kumo Forecast MASE | Seasonal-naive-12 | Timestral-28M univariate (Experiment 2) |
| --- | --- | --- | --- |
| NV | 0.515 | 0.150 | 0.151 |
| MI | 0.326 | 0.622 | 0.563 |
| CA | 0.214 | 0.656 | 0.466 |
| SD | 0.706 | 0.292 | 0.229 |

State unemployment, multivariate (one call, all four channels):

| State | Kumo Forecast MASE | Seasonal-naive-12 | Timestral-28M multivariate (Experiment 2) |
| --- | --- | --- | --- |
| NV | 0.301 | 0.150 | 0.156 |
| MI | 0.346 | 0.622 | 0.470 |
| CA | 0.200 | 0.656 | 0.071 |
| SD | 0.813 | 0.292 | 0.184 |

Beijing PM2.5 at Dongsi (Kumo Forecast's mean error was 114.0 µg/m³):

| Model | MASE | vs naive-24 (2.069) | vs naive-168 (1.791) |
| --- | --- | --- | --- |
| Kumo Forecast | 2.161 | loses | loses |
| Timestral-28M | 1.850 | beats | loses |
| Timestral-5M | 2.198 | loses | loses |

The pattern, as measured: Kumo Forecast is strong on Michigan and
California and poor on Nevada and South Dakota, where its univariate
MASE is about 3.4 and about 2.4 times the naive's respectively, and it
falls below both naive baselines on the hourly Beijing window. Where
Timestral beat the naive on three of four states, Kumo Forecast split
the panel.

The cost record needs care. A first attempt died silently mid-load,
with no traceback. A staged load test then measured the pieces:
checkpoint load took 5.5 s and reached 1,577 MB of memory; model build
took 14.3 s and reached 4,277 MB. At the time of the failed attempt,
unrelated compilation work on the same machine was holding about 4 GB,
so an out-of-memory kill during model build is the consistent
explanation. The rerun, after memory freed, completed: the first call
took 19.4 s including model load, later univariate calls took 1.2–1.5
s, the multivariate call took 4.3 s, the Beijing forecast 1.2 s, and
peak memory was 4,445.7 MB. A repeat call returned bit-identical
output. One build-time warning deserves a line: the backbone warns
that only its reconstruction head is pretrained. The Kumo checkpoint
then loaded over it with no missing and no unexpected keys, so the
warning describes the backbone's state before the checkpoint is
applied, not the final model.

## Experiment 4 — Kumo Anomaly on Beijing

Kumo Anomaly is the Kumo-TS repository's anomaly detector for
multivariate series, built on a diffusion model. The protocol followed
the repository's own example: `perform_anomaly_analysis_with_diffusion`
with the SCS threshold strategy and `nsample=15`, on Dongsi's 11
numeric channels (PM2.5, PM10, SO2, NO2, CO, O3, TEMP, PRES, DEWP,
RAIN, WSPM). The machinery delegates to a single-process inference
routine running a DPM-Solver with 20 steps. Each run warned "NaN
detected in observed_data!" and proceeded; the preprocessor is
NaN-aware, so the warning is part of normal operation on this data.

No attempt finished. The inference loop's progress indicator never
advanced past its initial state in any attempt, including under
unbuffered Python.

| Attempt | Input | Wall time | CPU time | Memory | Outcome |
| --- | --- | --- | --- | --- | --- |
| 1 | 1,440 rows (60 days) | about 21 min | about 25 min | about 865 MB | killed by a machine reboot; no output, no traceback |
| 2 | 1,440 rows (60 days) | 56 min 06 s | 51 min 15 s | 972 MB | still at the initial state; stopped by the probe; no output |
| 3 | 336 rows (14 days, 2013-08-13 to 2013-08-26) | 30 min 09 s, a hard bound | 43 min 22 s | 451 MB | stopped at the bound; no output |

One progress signal exists. In attempt 3, a second "NaN detected"
warning was logged 15 minutes 54 seconds after the first, which means
one full pipeline stage completed in about 16 minutes on 336 rows and
a second stage was under way. Stage cost scales with input length, so
at 1,440 rows each stage implies roughly an hour, with several stages
in the pipeline. Total compute burned across the three attempts was
about two hours of CPU time.

Because zero rows were scored in these attempts, the planned outputs
did not exist at that point: no score distribution, no anomaly fraction
under the SCS threshold, and no spot check of the highest-scored
windows. As of 2026-11-03, the result of this experiment was the
absence itself: at the repository's own example settings, CPU execution
on this machine did not finish a 14-day multivariate window in
30 minutes, nor a 60-day window in 56 minutes. The completion probe
below, run two days later, revises that result.

### Completion probe (2026-11-05): the detector finishes at reduced sampling

The stall above has a mechanical explanation. The SDK runs two
`evaluate()` passes over the input, and its progress indicator counts
batches while the whole input is a single batch: it reads `0it` until
sampling finishes, then shows the full elapsed time at once. A bar
that never advances is one unfinished batch, not a hang. Each pass
takes roughly a quarter of an hour on 336 rows at the example
sampling, which is why the 30-minute bound in attempt 3 fell inside
pass 2.

The probe reran the protocol at reduced sampling, `nsample=3`,
smallest first:

| Input | Inference wall | Peak memory | Outcome |
|---|---|---|---|
| 168 rows (7 days) | 512.9 s, uncontended | 566 MB | inference returned; the scores were then lost when the session failed to write its output file |
| 168 rows (7 days) | 2,557.5 s, under build contention | 488 MB | completed; scores persisted |
| 336 rows (14 days) | 2,582.1 s, under build contention | 674 MB | completed; scores persisted |

From the persisted runs: on 168 rows the per-row scores (mean absolute
reconstruction error) have median 0.290 and maximum 0.596, and the SCS
threshold flags 6 of 168 rows (3.6%). On 336 rows the median is
0.294, the maximum 0.562, and the threshold flags 15 of 336 (4.5%).
In both runs the five highest-scored rows are all SCS-flagged. Scores
are not inflated on rows containing a missing value: mean 0.300 on
such rows against 0.298 on complete rows (168 rows), and 0.307
against 0.303 (336 rows). The highest flags sit on a real episode:
ozone pinned at exactly 1071.0 for about nine consecutive hours on
2013-08-16 (robust z-score 11.7), a stuck or saturated sensor
plateau; the second-highest flag in the 336-row run is an ozone
collapse to 1.0 lasting hours after a missing hour. Some high flags
show no single-channel cause in the spot check, and the windows carry
no labels, so the record supports no precision or recall figure.

At the example settings the question stays open: `nsample=15` on the
168-row input did not finish within a 90-minute bound under load
(the first pass alone took 4,020 s with the machine under heavy
contention, so that figure is not comparable to the uncontended
runs). What the probe establishes is narrower: the detector
completes on this machine's CPU at `nsample=3`, in about 8.5 minutes
of uncontended inference for a 7-day window.

## Experiment 5 — Kumo Tabular against gradient boosting

Kumo Tabular is the single-table model in the same
`structured-data-models` library as Experiment 1: it predicts a label
for query rows from labeled context rows of one table. This experiment
supplies the comparison any predictive claim needs — the same task
given to an ordinary strong baseline, a gradient-boosted trees model,
on identical rows.

One data deviation, recorded: the corpus has no Taiwan credit-default
entry (its Taiwan entry is Taiwanese Bankruptcy, UCI 572), so the run
used the project's existing cleaned copy of the UCI 350 credit-default
dataset: 30,000 rows, 23 feature columns with the ID column excluded,
and the target column "default payment next month".

The protocol was one fixed split with seed 20261103: shuffled indices,
the first 4,000 rows as context and training data, the next 2,000 as
query and test. The context base rate was 0.2315 and the query base
rate 0.216. Kumo Tabular ran as `KumoTabular(task="classification",
size="small", device="cpu")`, fit on the context and predicting the
query; every feature was passed as a numerical semantic type — the
only feature type this model supports — and the target as categorical.
The baseline was scikit-learn's `HistGradientBoostingClassifier` at its
defaults, with `random_state` set to the same seed, trained on the
same 4,000 rows and the same raw features, with median imputation (the
data contained no missing values).

| Measurement | Kumo Tabular (small, CPU) | HistGradientBoosting |
| --- | --- | --- |
| AUROC on the 2,000-row query | 0.787 | 0.759 |
| Construction and checkpoint load | 50.9 s | — |
| Fit | 32.0 s | 1.78 s |
| Predict | 12.9 s | 0.014 s |
| Peak memory | 1,273 MB | (same process) |

Kumo Tabular finished ahead by 0.028 AUROC, at about 25 times the
baseline's fit-and-predict compute, excluding its 50.9 seconds of
construction and checkpoint load. Its predicted probabilities spanned
[0.026, 0.852]. The scope of the measurement is one task, one split,
and a defaults-only baseline: it records this pairing, and it is not a
general ranking of the two approaches.

## What the evaluation establishes

Each statement below is a measurement from the experiments above, with
its scope attached.

- **Kumo Relational runs on CPU through the open library.** Three
  probes produced sane, signal-bearing predictions at 728–745 MiB of
  peak memory, with forward passes of 4.43 and 8.26 seconds at context
  sizes of 400 and 1,200 rows. A GPU is not a requirement for the
  library path.
- **Timestral forecasts competitively on smooth monthly series and
  loses on a spiky hourly one.** On the unemployment panel, both
  checkpoints beat the lag-12 naive on three of four states in both
  modes, and the joint panel mode produced the best single cell of
  either round (28M multivariate California, MASE 0.071). On Beijing,
  both checkpoints lost to the weekly naive, and their intervals
  under-covered (0.521 and 0.438 against the 0.80 nominal level) where
  the panel intervals over-covered (1.000 in 13 of 16 cells).
- **On the identical protocol, Kumo Forecast is the weaker forecaster
  of the two on this data.** It split the panel — wins on Michigan and
  California, heavy losses on Nevada and South Dakota — and lost to
  both naives on Beijing. It reads only the last 512 context rows,
  zero-fills missing values where Timestral takes them as NaN, emits
  no quantiles, and peaked at 4,445.7 MB of memory, where Timestral's
  slowest single call took 0.52 seconds and its largest checkpoint
  loaded in 1.68 seconds.
- **Kumo Tabular beat a defaults gradient-boosting baseline on one
  fixed split** of Taiwan credit default, AUROC 0.787 against 0.759,
  at about 25 times the fit-and-predict compute.
- **Kumo Anomaly completes on CPU at reduced sampling, and not within
  the times attempted at its example settings.** At `nsample=3` it
  scored a 7-day window in 512.9 s of uncontended inference and a
  14-day window under contention, flagging 3.6% and 4.5% of rows; its
  highest-scored windows include a real stuck-sensor episode. At the
  example `nsample=15`, one pass on the smaller window took 4,020 s
  under load and the run did not finish within 90 minutes. On this
  hardware it is a usable screener only with its sampling reduced,
  and its flags carry no precision or recall figure, because the
  windows have no labeled ground truth.

## What the evaluation does not establish

- **A default Timestral checkpoint.** The 5M beat the 28M in 7 of 8
  panel cells, but the exception was the strongest single result in
  either round (multivariate California: 0.071 for the 28M against
  0.124 for the 5M), and on the hourly series the 28M lost by less
  than the 5M. The probes do not settle the choice, and the 123M and
  209M checkpoints were never run.
- **Kumo Relational's margin over ordinary practice.** The
  boosted-trees comparison ran for Kumo Tabular, not for the
  relational model. The relational evaluation the design calls for —
  graph recovery against declared schemas, plus holdout AUROC against
  boosted trees trained on flattened features — has not run.
  Experiment 1 shows the relational model reads planted signal and
  real channel structure; it does not say what it adds over flattened
  features and a standard model. The AdventureWorks AUROC of 1.000 is
  one near-determined task with a base rate of 0.802, not accuracy
  evidence.
- **Behavior past the observed context sizes.** The relational probes
  ran at 400 and 1,200 context rows, under a context cap the probes
  observed at roughly 2,000 rows. How labeled history beyond that size
  should be sampled or windowed is untested, and probe 2's AUROC drop
  under neighbor sampling is the warning that the choice moves the
  score.
- **Column kinds beyond the probed ones.** The probes exercised
  numeric, categorical, timestamp, and identifier columns. How text
  columns and index-role columns belong in a relational graph has no
  evidence behind it.
- **Ragged panels.** The panel probe used four states on one complete
  shared axis. Panels whose levels start and end at different times
  are untested.
- **Anything about other hardware.** Every timing and memory figure
  comes from one 4-core machine with 7,934 MiB of RAM, and Experiment
  3's first attempt showed that co-running work on the same machine
  can kill a model load. These numbers are a record of this
  environment, not a performance guarantee.
- **License clearance.** The Kumo weights ship under the OpenMDW 1.1
  license, and the Timestral weights under a Modified MIT license
  carrying a revenue cap. Neither license has been read in full for
  this project, so both model families are treated as external
  optional runtimes that a caller installs and points a screener at.
  Nothing of either is bundled.

- **Kumo Anomaly's flag quality.** The completion probe's spot check
  found its highest-scored rows on real sensor faults, but the Beijing
  windows carry no labels, so no precision or recall can be stated
  for its flags.

## Reproducibility index

The evaluation's working files are kept outside the repository, in
three directories, with the design note in a fourth. This appendix was
written from those files, and every number above traces to one of them.

- **Round 1, Kumo Relational** — `~/workspace/research_notes/kumo-probe-20261103/`:
  `RUN_LOG.md` (the full run log), `probe_relational.py`,
  `probe_sampler.py`, `probe_adventureworks.py`, the memory-wrapper
  runners (`run_probe.py`, `run_sampler.py`, `run_aw.py`), and the
  install and download transcripts (`install*.log`, `dl-torch.sh` and
  its log, `dl-ckpt.sh` and its log).
- **Round 1, Timestral** — `~/workspace/research_notes/timestral-probe-20261103/`:
  `SUMMARY.md`, `results.json` (the raw numbers behind every Experiment
  2 table), `run_probe.py`, the install logs, `hf_download.log`, and
  `run3.log` (the final run; `run.log` and `run2.log` are the aborted
  attempts).
- **Round 2, Kumo-TS** — `~/workspace/research_notes/kumo-ts-probe-20261103/`:
  `SUMMARY.md`, `results_leg1.json`, `results_leg2_7d.json` and
  `results_leg2_14d.json` with their score files
  (`beijing_anomaly_scores_7d.csv`, `beijing_anomaly_scores_14d.csv`,
  and the stage score archives — all from the 2026-11-05 completion
  probe in Experiment 4), and `results_leg3.json`,
  the experiment scripts (`prep_data.py`, `leg1_forecast.py`,
  `leg1_loadtest.py`, `leg2_anomaly.py`, `leg2_anomaly_14d.py`,
  `leg2_spotcheck.py`, `leg3_tabular.py`), the prepared inputs
  (`beijing_multi.csv`, `beijing_multi_14d.csv`, `beijing_pm25.npz`,
  `unemp.npz`), `leg3_predictions.npz`, the run logs, and the `weights`
  directory.
- **Design note** — `~/workspace/research_notes/onceover-foundation-screeners-20261103/design.md`:
  the screener design these experiments feed, including the evaluation
  each screener must still pass before its statements ship.